# Data Classification - Unstructured Data

This notebook represents a simple example of the utilization of the READI framework. In this series of example we will focus on data classification, in particular we will focus on unstructured data.

This include natural language, also known as free text, and loosely formatted/structured documents. For example JSON/XML document without schema, or with heterogeneous schema.

In this context, we refer to data classification as the ability to assign semantic values to textual component, generally a span within a text.


## Installation

The installation of the framework can be performed simply via `pip install`, as here demonstrated:

## 1. Simple scenario: READI as a blackbox

Let's go through a more "off the shelves" example, where we use PII/PHI functionality on large datasets.

In this case we use [`gretelai/gretel-pii-masking-en-v1`](https://huggingface.co/datasets/gretelai/gretel-pii-masking-en-v1), a public dataset available in HuggingFace, and we will leverage predefined set of tools and configuration.

More precisely, we are extracting 100 random data points from the `train` partition.

In [ ]:
import datasets

data = datasets.load_dataset("gretelai/gretel-pii-masking-en-v1", split="train").shuffle()[:100]  # type: ignore

In the most basic way to use READI is the following:

In [ ]:
from risk_assessment.readi.analyzer import READIAnalyzer

readi = READIAnalyzer()

for text in data["text"]:
    print(20 * "-")
    print(text)
    print(7 * "-" + "RESULT" + 7 * "-")
    [
        print(f"{entity.entity_type} {','.join(entity.source)} -> {text[entity.start:entity.end]}")
        for entity in readi.detect(text)
    ]
    print("\n")

## 2. One step further: explicit usage of extractors and aggregator

For simplicity, we use pre-defined extractors, which could be found in the utility package `risk_assessment.readi.defaults_phi`, and a default configuration

In [ ]:
from risk_assessment.readi.defaults_phi import AGGREGATOR_CONFIGURATION, ENTITY_EXTRACTORS

In the case of unstructured document READI will need to instantiate a certain number of instsances of `EntityExtractor` (>=1) and one `Aggregator`.
The latter will take care of processing the entities detected by various annotators and disambiguate between the various identified types.

In fact, these defaults are a list of classes extending `EntityExtractor`, which will extract entities from the data according to their characteristics.
They also include the configuration for an `Aggregator`.

In this simple example, we also show how extend the set of available extractors by using a Spacy model, also showing how to customize the source of the annotation by replacing its default name (`Spacy`) with a mnemonic reference, say `SPACY_ONTONOTES` to indicate the dataset used in trainig.

On the other hand, let us use the aggregator configuration as-is.

In [ ]:
from risk_assessment.classification.unstructured.aggregator import Aggregator
from risk_assessment.classification.unstructured.spacy import SpacyEntityExtractor

extractors = list(ENTITY_EXTRACTORS) + [SpacyEntityExtractor("en_core_web_lg", extractor_name="SPACY_ONTONOTES")]
aggregator = Aggregator(AGGREGATOR_CONFIGURATION)

We can now process the content of the dataset as follows:

In [ ]:
for text in data["text"]:
    print(20 * "-")
    print(text)
    print(20 * "-")
    print(7 * "-" + "RESULT" + 7 * "-")
    [
        print(f"{entity.entity_type} {','.join(entity.source)} -> {text[entity.start:entity.end]}")
        for entity in aggregator.aggregate([extractor.extract(text) for extractor in extractors], text)
    ]
    print("\n")

## 3. A more complex scenario: under the hood

In a more complext scenario, let us decide to customize the configuration for the aggregator and to select the entity extractors.
For example, as follow:
1. `DRLEntityExtractor`, which is based on the dictionary and regular expression presented before,
2. `SpacyEntityExtractor`, and
3. `STANZAEntityExtractor`, to leverage the state of the art model based NER technology.

Moreover, we might want to use multiple instances of each `EntityExtractor` to allow a more fine grained detection of types, basically merging multiple models from the same provider, and using more tokenizers. For this reason, we download two Spacy models.
The first one is `xx_ent_wiki_sm`, which is a small multi lingual model, and `en_core_web_lg` is a model specialized for English.

In [ ]:
# optional step, the SpacyEntityExtractor object can automatically download models if not already available
# %pip install https://github.com/explosion/spacy-models/releases/download/xx_ent_wiki_sm-3.7.0/xx_ent_wiki_sm-3.7.0-py3-none-any.whl


In [ ]:
from risk_assessment.classification.identifiers import (
    IBAN,
    IMEI,
    IP,
    SSN,
    SSNUK,
    URI,
    CreditCard,
    DateTime,
    Email,
    NationalIdentity,
    Phone,
    UKPostCode,
    USPostalAddress,
    ZipCode,
)
from risk_assessment.classification.unstructured import EntityExtractor, MultiSourceEntityExtractor
from risk_assessment.classification.unstructured.drl import DRLEntityExtractor
from risk_assessment.classification.unstructured.spacy import SpacyEntityExtractor
from risk_assessment.classification.unstructured.stanza_ner import STANZAEntityExtractor
from risk_assessment.readi.external_tokenizers import SpacyTokenizer

entity_extractors: list[EntityExtractor | MultiSourceEntityExtractor] = [
    SpacyEntityExtractor(
        "xx_ent_wiki_sm",  # let us use a multilingual spacy model
        type_mapping={
            "PER": "Person",  # the type mapping allows us to create a uniform set of entity name between various models
            "PERSON": "Person",  # here, for example we instruct the EntityExtractor to convert the label "PERSON" to "Person"
            "GPE": "Location",
            "LOC": "Location",
            "DATE": "Date",
            "NORP": "Person",
            "TIME": "Date",
            "FAC": "Location",
            "ORG": "Organization",
        },
        extractor_name="SPACY_ML",
    ),  # that we will refer later on as "SPACY_ML"
    SpacyEntityExtractor(
        "en_core_web_lg",  # On the other hand, we here use the tranformer-based model for English
        type_mapping={
            "PERSON": "Person",
            "GPE": "Location",
            "LOC": "Location",
            "DATE": "Date",
            "NORP": "Person",
            "TIME": "Date",
            "FAC": "Location",
            "ORG": "Organization",
        },
    ),  # since we are not specifying a source name, it will be referred as "SPACY"
    STANZAEntityExtractor(
        type_mapping={
            "PERSON": "Person",  # note that we are not specifying a specific model to use, hence using the default one (english)
            "GPE": "Location",
            "LOC": "Location",
            "DATE": "Date",
            "NORP": "Person",
            "TIME": "Date",
            "FAC": "Location",
            "ORG": "Organization",
        }
    ),  # similarly, this one will be referred as "STANZA"
    DRLEntityExtractor(
        identifiers=[
            IMEI(),
            CreditCard(),
            IBAN(),
            SSN(),
            SSNUK(),
            DateTime(),
        ],  # we here use a DRLEntityExtractor, with a specific list of identifiers enabled
        max_shinglet_length=15,  # and we are specifying a shinglet lenght different form the default one (5)
        type_mapping={"DateTime": "Date"},
    ),  # as per the other EntityExtractor, even here one can specify a type_mapping
    DRLEntityExtractor(
        identifiers=[IMEI(), CreditCard(), IBAN(), SSN(), SSNUK(), DateTime(), Phone(), IP(), URI(), Email()],
        tokenizer=SpacyTokenizer(),  # this DRLEntityExtractor leverages a different tokenizer, to take fully advantage of the Aggregator's ability to reconciliate different spans-type pairs
        type_mapping={"DateTime": "Date"},
        max_shinglet_length=15,
    ),
    DRLEntityExtractor(identifiers=[Email(), URI()], max_shinglet_length=15),
    DRLEntityExtractor(
        identifiers=[IP(), Phone(), ZipCode(), NationalIdentity(), UKPostCode(), USPostalAddress()],
        type_mapping={"DateTime": "Date"},
        max_shinglet_length=15,
    ),
]

Let us now define an `Aggregator`, which will reconciliate the various `EntityExtractors` and bring order to the chaos.

In [ ]:
from risk_assessment.classification.unstructured.aggregator import Aggregator, AggregatorConfiguration

aggregator = Aggregator(
    AggregatorConfiguration(
        prioritize_inclusion=False,
        merge_entities=True,
        weights={
            "UKPostCode": {"DRL": 1200},
            "USPostalAddress": {"DRL": 1200},
            "IP": {"DRL": 2000},
            "ZipCode": {"DRL": 100},
            "SSN": {"DRL": 2000},
            "SSNUK": {"DRL": 2000},
            "Location": {"DRL": 1300, "SPACY": 500, "STANZA": 500},
            "Person": {"STANZA": 1000, "SPACY": 450, "SPACY_ML": 450, "DRL": 100},
            "Phone": {"DRL": 1000},
            "URI": {"DRL": 400},
            "NationalIdentity": {"DRL": 1500},
            "Date": {"DRL": 1000, "STANZA": 450, "SPACY": 450},
            "Email": {"DRL": 2000},
            "IBAN": {"DRL": 4000},
            "CreditCard": {"DRL": 1000},
            "IMEI": {"DRL": 1000},
        },
        thresholds={
            "Person": 900.0,
            "Date": 900.0,
        },
        to_report_only={
            "Person",
            "Location",
            "Organizarion",
            "Date",
            "Email",
            "USPostalAddress",
            "UKPostCode",
            "ZipCode",
            "IBAN",
            "CreditCard",
            "IMEI",
            "IP",
            "URI",
            "Phone",
            "NationalIdentity",
            "SSN",
            "SSNUL",
        },
    )
)

Let us also assume that the following is the text that needs to be processes, which is a dump of the Wikipedia page for Jacqueline Kennedy.

In [ ]:
document_content = """Who Was Jacqueline Kennedy Onassis?
                Jacqueline Kennedy Onassis married John F. Kennedy in1953. When she became first lady in 1961, she worked to restore the White House to its original elegance and to protect its holdings. After JFK's assassination in 1963, she moved to New York City and married Aristotle Onassis in 1968. She died of cancer in 1994.
                Early Life
                Jacqueline Bouvier Kennedy Onassis was born on July 28, 1929, in Southampton, New York. Her father, John Bouvier, was a wealthy New York stockbroker of French Catholic descent, and her mother, Janet, was an accomplished equestrienne of Irish Catholic heritage. Onassis was a bright, curious and occasionally mischievous child. One of her elementary school teachers described her as "a darling child, the prettiest little girl, very clever, very artistic, and full of the devil." Another teacher, less charmed by young Jacqueline, wrote admonishingly that "her disturbing conduct in geography class made it necessary to exclude her from the room."
                Onassis enjoyed a privileged childhood of ballet lessons at the Metropolitan Opera House and French lessons beginning at age of 12. Like her mother, Onassis loved riding and was highly skilled on horseback. In 1940, at the age of 11, she won a national junior horsemanship competition. The New York Times reported, "Jacqueline Bouvier, an eleven-year-old equestrienne from East Hampton, Long Island, scored a double victory in the horsemanship competition. Miss Bouvier achieved a rare distinction. The occasions are few when the same rider wins both competitions in the same show. Onassis attended Miss Porter's School, a prestigious boarding school in Farmington, Connecticut; in addition to its rigorous academics, the school also emphasized proper manners and the art of conversation. There she excelled as a student, writing frequent essays and poems for the school newspaper and winning the award as the school's top literature student in her senior year. Also during her senior year, in 1947, Onassis was named "Debutante of the Year" by a local newspaper. However, Onassis had greater ambitions than being recognized for her beauty and popularity. She wrote in the yearbook that her life ambition was "not to be a housewife."
                Upon graduating from Miss Porter's School Onassis enrolled at Vassar College in New York to study history, literature, art and French. She spent her junior year studying abroad in Paris. "I loved it more than any year of my life," Onassis later wrote about her time there. "Being away from home gave me a chance to look at myself with a jaundiced eye. I learned not to be ashamed of a real hunger for knowledge, something that I had always tried to hide, and I came home glad to start in here again but with a love for Europe that I am afraid will never leave me."
                Upon returning from Paris, Onassis transferred to George Washington University in Washington, D.C., and graduated with a B.A. in French literature in 1951. After graduating from college in 1951, Onassis landed a job as the "Inquiring Camera Girl" for the Washington Times-Herald newspaper. Her job was to photograph and interview various Washington residents, and then weave their pictures and responses together in her column. Among her most notable stories were an interview with Richard Nixon, coverage of President Dwight D. Eisenhower's inauguration and a report on the coronation of Queen Elizabeth II.
                U.S. First Lady
                It was at a dinner party in 1952 that Onassis met a dashing young congressman and senator-elect from Massachusetts named John F. Kennedy; he "leaned across the asparagus and asked her for a date." They were married a year later, on September 12, 1953. Onassis gave birth to her first child, Caroline Kennedy, in 1957. That same year, she encouraged Kennedy to write and, subsequently, helped him edit Profiles in Courage, his famous book about U.S. senators who had risked their careers to stand for causes they believed in.
                In January 1960, John F. Kennedy announced his candidacy for the U.S. presidency. Although Onassis was pregnant at the time and thus unable to join him on the campaign trail, she campaigned tirelessly from home. She answered letters, gave interviews, taped commercials and wrote a weekly syndicated newspaper column called "Campaign Wife."
                On November 8, 1960, Kennedy defeated Richard Nixon by a razor-thin margin to become the 35th president of the United States; less three weeks later, Onassis gave birth to their second child, John Fitzgerald Kennedy Jr. The couple had a third child, Patrick Bouvier Kennedy born prematurely on August 7, 1963, but lost the child two days later.
                Onassis's first mission as first lady was to transform the White House into a museum of American history and culture that would inspire patriotism and public service in those who visited. "Every boy who comes here should see things that develop his sense of history," she once said. Onassis went to extraordinary lengths to procure art and furniture owned by past presidents—including artifacts owned by George Washington, James Madison and Abraham Lincoln—as well as pieces she considered representative of various periods of American culture. "Everything in the White House must have a reason for being there," she insisted. "It would be sacrilege merely to 'redecorate' it—a word I hate. It must be restored—and that has nothing to do with decoration. That is a question of scholarship."
                As the culmination of her project, Onassis gave a tour of the restored White House on national television on February 14, 1962. A record 56 million viewers watched her televised special, and Onassis won an honorary Emmy Award for her performance.
                Assassination of JFK
                On November 22, 1963, Onassis was riding alongside the president in a Lincoln Continental convertible before cheering crowds in Dallas, Texas, when he was shot and killed by Lee Harvey Oswald, widowing Onassis at the age of 34. The first lady's stoic composure in her bloodstained pink suit became the symbol of national mourning. It was also Onassis who, in the aftermath of the president's death, provided a metaphor for her husband's administration that has remained its enduring symbol: Camelot, the idyllic castle of the legendary King Arthur. "There'll be great presidents again," Onassis said, "but there'll never be another Camelot again.
                Marriage to Aristotle Onassis
                In 1968, five years after John F. Kennedy's death, Onassis married a Greek shipping magnate named Aristotle Onassis. However, he died only seven years later, in 1975, leaving Onassis a widow for the second time.
                Following the death of her second husband, Onassis returned to the promising career that had been put on hold when she married Kennedy. She went to work as an editor at the Viking Press in New York City and then moved to Doubleday, where she served as senior editor.
                Jacqueline Bouvier Kennedy Onassis died on May 19, 1994, at the age of 64. She is buried beside President John F. Kennedy's gravesite at the Arlington National Cemetery, which is marked by the eternal flame."""

The processing is fairly simple, by running the entity extrators and piping their output to the aggregator.

In [ ]:
extractor_outputs = [extractor.extract(document_content) for extractor in entity_extractors]

for output, source in zip(extractor_outputs, entity_extractors, strict=False):
    print(type(source), f"{len(output)} entities detected")

Note how several entities are detected, but most likely not all of them are relevant, or correct.

In [ ]:
aggregared_entities = aggregator.aggregate(extractor_outputs, document_content)

for entity in aggregared_entities:
    print(f"{document_content[entity.start:entity.end]} ->", f"{entity.entity_type} {','.join(entity.source)}")